## Elaboración de la base de datos
Propuesta de Investigación
- Curso: Estadística para el Análisis Político 2
- Nombres: Estefanía Apaza (20230487) y Diego Luyo (20230934)


Insumos

[Libro de Códigos](https://docs.google.com/spreadsheets/d/1U5zJE58q_83H3Lc0-il9jx8QrDb9EX78/edit?gid=266433641#gid=266433641)

[Base de Eventos de Protesta](https://github.com/estefania-apaza/state-capacity-protest-peru/blob/main/Base%20de%20Eventos%20de%20Protesta%20del%20Peru%CC%81_1980_2025.csv) (Aragón, et al.) - Necesario subirla a Google Colab para correr el código

### Creación de la variable Y

Limpieza y filtrado de la base de datos

In [48]:
import pandas as pd

# Cargado de la base de la Escuela de Gobierno y Políticas Públicas PUCP

# Tu enlace de Google Sheets publicado como CSV
link_egpp = "https://docs.google.com/spreadsheets/d/e/2PACX-1vTmmfwRdE8OQdDmTi1Tj2Y7gmT2BtwD19Q170SIMn9gMyjK7djXpjHofQ-4Lu0xB5-Xs29WI2PL-Lzq/pub?gid=1381686519&single=true&output=csv"

df_egpp = pd.read_csv(link_egpp, encoding='utf-8')

# Limpieza de nombres de columnas
df_egpp.columns = df_egpp.columns.str.strip().str.lower()
df_egpp.columns = df_egpp.columns.str.replace('ñ', 'n').str.replace('á', 'a').str.replace('é', 'e').str.replace('í', 'i').str.replace('ó', 'o').str.replace('ú', 'u')

# Verificamos que cargó mostrando las primeras filas
print(f"✅ Base cargada. Filas: {len(df_egpp)}")
df_egpp.head(5)

✅ Base cargada. Filas: 25026


,id,periodico,periodico_id,dia,mes_id,mes,ano,fecha,presidente_id,presidente,...,institucion_1,institucion_2,institucion_3,institucion_id,reclamo_t,reclamo,reclamo_id,sub_reclamo,sub_reclamo_id,comentar_t
0,1,Expreso,5.0,1,1,Enero,1980,############,1,Morales Bermúdez,...,Empresas periodísticas,Periodicos y diarios.,NaN,909.0,Exigen que las empresas periodísticas les otor...,Laborales,1,Bonificaciones,105,NaN
1,2,Expreso,5.0,3,1,Enero,1980,############,1,Morales Bermúdez,...,Instituto Peruano de Seguridad Social,NaN,NaN,712.0,Demandan una mejora en condiciones salariales.,Laborales,1,Aumentos salariales,101,"Exigen además: recategorización, aumento gener..."
2,3,Expreso,5.0,4,1,Enero,1980,############,1,Morales Bermúdez,...,Ministerio de Justicia,Centro de Readaptación Social de Chorrillos,NaN,105.0,Protestan por la mala alimentación recibida en...,Servicios,4,Alimentación,407,15 internas resultaron heridas.
3,4,Expreso / El Comercio,6.0,4,1,Enero,1980,############,1,Morales Bermúdez,...,Empresas periodísticas,Empresa Periodistica Nacional S.A.,NaN,909.0,Protestan contra medidas de la empresa que ate...,Laborales,1,Mejores condiciones laborales,104,Huelga declarada ilegal por el gobierno. Inici...
4,5,El Comercio,7.0,4,1,Enero,1980,############,1,Morales Bermúdez,...,Municipalidades Provinciales de Cusco,Cusco,NaN,508.0,Denuncian el despido arbitrario de trabajadores.,Laborales,1,Estabilidad laboral,109,También demandan la destitución de cuatro func...


In [49]:
# Selección preliminar de columnas
columnas_analisis = [
    'id', 'ano', 'mes_id','presidente', 'presidente_id'
    'region_id', 'region', 'provincia_id', 'provincia', 'distrito_id','distrito', 'sector_1',
    'sector_id_1', 'actor_1', 'actor_1_id', 'sector_id_2', 'actor_2_id',
    'accion_1_id', 'accion_2_id', 'accion_3_id', 'accion_4_id',
    'duracion_horas', 'numero_participantes',
    'numero_heridos', 'numero_muertos', 'numero_detenidos','adversario',
    'adversario_id', 'institucion_1', 'institucion_id', 'reclamo',
    'reclamo_id', 'sub_reclamo_id']

# Creación de la base preliminar
df_preliminar = df_egpp[[c for c in columnas_analisis if c in df_egpp.columns]].copy()

df_preliminar.columns
df_preliminar.head(5)

,id,ano,mes_id,presidente,region,provincia_id,provincia,distrito_id,distrito,sector_1,...,numero_heridos,numero_muertos,numero_detenidos,adversario,adversario_id,institucion_1,institucion_id,reclamo,reclamo_id,sub_reclamo_id
0,1,1980,1,Morales Bermúdez,Lima,1501,Lima,150101,Lima,Comercial,...,NaN,NaN,NaN,Empresas Privadas,9,Empresas periodísticas,909.0,Laborales,1,105
1,2,1980,1,Morales Bermúdez,Lima,1501,Lima,150101,Lima,Salud,...,NaN,NaN,NaN,Organismos Autónomos,7,Instituto Peruano de Seguridad Social,712.0,Laborales,1,101
2,3,1980,1,Morales Bermúdez,Lima,1501,Lima,150108,Chorrillos,Personas privadas de libertad,...,15.0,NaN,NaN,Poder Ejecutivo,1,Ministerio de Justicia,105.0,Servicios,4,407
3,4,1980,1,Morales Bermúdez,Lima,1501,Lima,150101,Lima,Prensa,...,NaN,NaN,NaN,Empresas Privadas,9,Empresas periodísticas,909.0,Laborales,1,104
4,5,1980,1,Morales Bermúdez,Cusco,801,Cusco,80101,Cusco,Municipal,...,NaN,NaN,NaN,Gobierno Provincial,5,Municipalidades Provinciales de Cusco,508.0,Laborales,1,109


**Definición del umbral de la violencia para la variable dependiente: "Protesta violenta"**

Criterios

*   Alguna de las acciones implica violencia según el Libro de Códigos
*   Existe alguna víctima física



In [50]:
def variable_violencia(row):
    # Códigos de Acción seleccionados por implicar violencia según el Libro de Códigos (107-114, 119, 123)
    codigo_violencia = [107, 108, 109, 110, 111, 112, 113, 114, 119, 123]

    # Extraemos los IDs mencionados de las columnas de Acción (1-4)
    acciones = [row.get('accion_1_id'), row.get('accion_2_id'),
                row.get('accion_3_id'), row.get('accion_4_id')]

    # Criterio 1 - Acción implica violencia
    es_accion_violenta = any(acc in codigo_violencia for acc in acciones)

    # Criterio 2 - Existen víctimas físicas
    muertos = pd.to_numeric(row.get('numero_muertos', 0), errors='coerce') or 0
    heridos = pd.to_numeric(row.get('numero_heridos', 0), errors='coerce') or 0
    tiene_victimas = (muertos > 0 or heridos > 0)

    if es_accion_violenta or tiene_victimas:
        return 1
    return 0

# Creamos la variable dependiente con la función
df_preliminar['violencia_y'] = df_preliminar.apply(variable_violencia, axis=1)

print(df_preliminar['violencia_y'].value_counts())

violencia_y
0    21144
1     3882
Name: count, dtype: int64


### Creación de la variable X

**Definición de periodos de oportunidad política**

*  Bloque 1 (Pre-90): Años menores a 1990.
*  Bloque 2 (90-00): Desde 1990 hasta el 2000.
*  Bloque 3 (00-16): Del 2001 al 2016.
*  Bloque 4 (Post-2016): Del 2017 en adelante.

In [51]:
# Definición de la función usando la variable de año [ano]

def variable_x(row):
    # Extraemos el año de la fila
    ano = row.get('ano')
    
    # Iniciamos con los N/A
    if pd.isna(ano):
        return None

    # Evaluamos los bloques con las condiciones propuestas
    if ano < 1990:
        return 1  # Bloque 1: Pre-90

    elif 1990 <= ano <= 2000:
        return 2  # Bloque 2: 90-00

    elif 2001 <= ano <= 2016:
        return 3  # Bloque 3: 00-16

    else:
        return 4  # Bloque 4: Post-2016 (Años mayores o iguales a 2017)

# Aplicamos la función fila por fila (axis=1) para crear la variable_x (Periodo Político)
df_preliminar['periodo_politico'] = df_preliminar.apply(variable_x, axis=1)

# Verificación de eventos por bloque
df_preliminar['periodo_politico'].value_counts().sort_index()


periodo_politico
1     6995
2     3377
3    10399
4     4255
Name: count, dtype: int64

### Variables de control


##### 1. Protesta masiva

In [52]:
# Evaluamos corte para protesta masiva
df_preliminar['numero_participantes'].quantile([0.25, 0.5, 0.75, 0.9, 0.95])

0.25      150.0
0.50      600.0
0.75     3500.0
0.90    12000.0
0.95    30000.0
Name: numero_participantes, dtype: float64

In [53]:
# Seleccionamos el umbral
umbral_elegido = 3500

# Elaboramos la variable
df_preliminar['protesta_masiva'] = (df_preliminar['numero_participantes'] >= umbral_elegido).astype(int)

print(df_preliminar['protesta_masiva'].value_counts())


protesta_masiva
0    24088
1      938
Name: count, dtype: int64


##### 2. Número de eventos

In [54]:
# Agrupamos para contar cuántas protestas hubo por provincia cada mes
conteo_mensual = df_preliminar.groupby(['provincia_id', 'ano', 'mes_id']).size().reset_index(name='n_eventos_mes')

# Ordenamos cronológicamente por provincia, año y mes
conteo_mensual = conteo_mensual.sort_values(['provincia_id', 'ano', 'mes_id'])

# Desplazamos el valor: el conteo del mes anterior se asigna al mes actual
conteo_mensual['numero_eventos_previos'] = conteo_mensual.groupby('provincia_id')['n_eventos_mes'].shift(1).fillna(0)

# Realizamos el merge limpiando el dataframe derecho
df_preliminar = pd.merge(
    df_preliminar,
    conteo_mensual[['provincia_id', 'ano', 'mes_id', 'numero_eventos_previos']].drop_duplicates(),
    on=['provincia_id', 'ano', 'mes_id'],
    how='left'
)

# Si alguna provincia no tuvo protestas el mes anterior, volvemos el NA un 0
df_preliminar['numero_eventos_previos'] = df_preliminar['numero_eventos_previos'].fillna(0).astype(int)

print(df_preliminar['numero_eventos_previos'].describe())

count    25026.000000
mean        14.437105
std         18.704450
min          0.000000
25%          1.000000
50%          5.000000
75%         21.000000
max        122.000000
Name: numero_eventos_previos, dtype: float64


##### 3. Tipo de actor

In [55]:
# El ID del actor debe ser número entero
df_preliminar['actor_1_id'] = df_preliminar['actor_1_id'].astype(int)

# Creamos una función para clasificar según los IDs oficiales del libro de códigos
def clasificar_por_id(row):
    act_id = row['actor_1_id']
    
    # --- 1. Gremios Laborales / Sindicatos ---
    # Docentes, enfermeras, médicos, bancarios, trabajadores estatales, centrales sindicales (CGTP, CITE, etc.)
    if act_id in [103, 104, 105, 106, 115, 116, 117, 118, 119, 122, 125, 126, 131, 132, 133, 205, 207, 301, 302, 303, 304, 305]:
        return 'actor_laboral'
        
    # --- 2. Organizaciones Territoriales / Vecinales / Comunales ---
    # Rondas, frentes de defensa, comunidades campesinas y nativas, comités de regantes, pobladores y vecinos
    elif act_id in [124, 201, 208, 209, 210, 211, 212, 401]:
        return 'actor_territorial_social'
        
    # --- 3. Organizaciones Económicas / Corporativas ---
    # Cocaleros, comerciantes, pescadores, mineros, transportistas, agricultores, ganaderos, empresarios, colectiveros
    elif act_id in [101, 102, 107, 108, 109, 110, 111, 112, 113, 114, 120, 121, 127, 128, 129, 130, 204]:
        return 'actor_economico'
        
    # --- 4. Organizaciones Estudiantiles / Educativas ---
    # Estudiantes de universidades y colegios (Los separamos por su dinámica particular)
    elif act_id in [202, 203]:
        return 'actor_estudiantil'
        
    # --- 5. Colectivos Ciudadanos / ONGs / Activistas (Categoría de Referencia) ---
    # Colectivos, asociaciones civiles, comités, ONGs, padres de familia, y los grupos vulnerables de Diego
    # (Damnificados, licenciados de FFAA, discapacitados, víctimas de violencia, comunidad LGTB, usuarios)
    else:
        return 'actor_politico_ciudadano'

# Aplicamos la función para crear la columna de texto
df_preliminar['categoria_actor'] = df_preliminar.apply(clasificar_por_id, axis=1)

# Creamos las variables Dicotómicas (Dummies 0 o 1) con Pandas
dummies_actores = pd.get_dummies(df_preliminar['categoria_actor'], prefix='', prefix_sep='', dtype=int)

# Unión de los dummies a la base preliminar para crear la base final
df_preliminar = pd.concat([df_preliminar, dummies_actores], axis=1)
df_preliminar = df_preliminar.drop(columns=['categoria_actor'])

# Verificación de la creación de las variables Dicotómicas de Actores
print("--- VARIABLES DICOTÓMICAS DE ACTORES CREADAS ---")
columnas_actores = ['actor_laboral', 'actor_territorial_social', 'actor_economico', 'actor_estudiantil', 'actor_politico_ciudadano']
for col in columnas_actores:
    print(f"Variable {col}: {df_preliminar[col].sum()} filas marcadas con 1.")

--- VARIABLES DICOTÓMICAS DE ACTORES CREADAS ---
Variable actor_laboral: 6913 filas marcadas con 1.
Variable actor_territorial_social: 6234 filas marcadas con 1.
Variable actor_economico: 4550 filas marcadas con 1.
Variable actor_estudiantil: 1558 filas marcadas con 1.
Variable actor_politico_ciudadano: 5771 filas marcadas con 1.


### Creación de la base final

In [56]:
print("--- REVISIÓN DE DISTRIBUCIÓN VARIABLE Y ---")
print(f"Total de filas: {len(df_preliminar)}")
print(f"Violencia (Y):\n{df_preliminar['violencia_y'].value_counts()}\n")

--- REVISIÓN DE DISTRIBUCIÓN VARIABLE Y ---
Total de filas: 25026
Violencia (Y):
violencia_y
0    21144
1     3882
Name: count, dtype: int64



In [57]:
# Selección de columnas final

columnas_final = [
    'id', 'ano', 'mes_id','presidente', 'presidente_id'
    'region_id', 'region', 'provincia_id', 'provincia', 'distrito_id','distrito', 'sector_1',
    'sector_id_1', 'actor_1', 'actor_1_id', 'sector_id_2', 'actor_2_id',
    'duracion_horas', 'numero_participantes','numero_detenidos','adversario',
    'adversario_id', 'institucion_1', 'institucion_id', 'reclamo',
    'reclamo_id', 'sub_reclamo_id', 'violencia_y', 'periodo_politico', 'protesta_masiva',
    'numero_eventos_previos', 'actor_laboral', 'actor_territorial_social', 'actor_economico',
    'actor_estudiantil', 'actor_politico_ciudadano']

# Creación de la base final para la variable Y
df_final = df_preliminar[[c for c in columnas_final if c in df_preliminar.columns]].copy()

Descarga de la base de datos (desbalanceada)

In [58]:
df_final.to_csv('base_desbalanceada_limpia.csv', index=False, encoding='utf-8-sig')

**Referencias**

Aragón, Jorge, Moisés Arce, Renzo Aurazo y Omar Coronel. 2025. Base de Eventos de Protestas del Perú, Versión Agosto 2025. Lima: Pontificia Universidad Católica del Perú-PUCP

Aragón, Jorge, Moisés Arce, Renzo Aurazo y Omar Coronel. 2025. Base de Eventos de Protestas del Perú: Libro de Códigos, Versión Agosto 2025. Lima: Pontificia Universidad Católica del Perú-PUCP